# 1. Data and labels

What the models learn from. Everything here is read from committed files; rebuild them with `make data` and `make labels`.

In [ ]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # notebook-only: inline display works without it too
import matplotlib.pyplot as plt
import pandas as pd

from hotel_curator.config import Settings

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
settings = Settings(home=ROOT)

In [ ]:
import json

data = json.loads((settings.manifests_dir / "data.json").read_text())
pd.Series(data["stats"]["hotels_per_city"], name="hotels").to_frame()

In [ ]:
from hotel_curator.ml.data import load_all
from hotel_curator.taxonomy import LABELS

sets = load_all(settings.labels_dir)
prevalence = pd.DataFrame(
    {name: s.frame[list(LABELS)].mean() for name, s in sets.items()}
)
prevalence.round(3)

## How much do the annotators agree, and how do they compare with people?

In [ ]:
agreement = pd.DataFrame(
    json.loads((settings.labels_dir / "agreement.json").read_text())
)
agreement.pivot(
    index="label", columns="set", values="fleiss_kappa"
).round(3)

In [ ]:
external = json.loads(
    (settings.labels_dir / "external_check.json").read_text()
)
pd.DataFrame(external["explicit_implicit_triple_kappa"]).set_index(
    "label"
)[["human_human_kappa", "panel_human_kappa"]].round(2)